#### Modes :
- **APPEND** : It just processes and appends the new data. It does not allow you to perform Stateful Transformation like Aggregation. Aggregation is basic and in real world it will be required.
- **COMPLETE** : Suitable for stateful transformations. Executor keep track of previous state of data when processes new data. There is Source , OutPut, and Destination. Output is nothing but state created by executors everytime new data is processed with the help of cached data and final data in Destination is kind of a overwrite only from Output.
- **UPDATE** : Here that value or executor will not be part of output which is not updated. Only Updated data is updated in Destination Location.

Note : To understand this with example it will be good to have delta table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbcertification.streaming.sourcetable
(
    color string
)
USING DELTA
LOCATION 'abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sourcetable'

In [0]:
%sql
insert into dbcertification.streaming.sourcetable values
    ('red'),
    ('green'),
    ('blue'),
    ('yellow'),
    ('orange'),
    ('orange')

In [0]:
%sql
select * from dbcertification.streaming.sourcetable;

In [0]:
df = spark.readStream.table("dbcertification.streaming.sourcetable")

In [0]:
%skip
CREATE TABLE IF NOT EXISTS dbcertification.streaming.sinktable
(
    color string
)
USING DELTA
LOCATION 'abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable'

In [0]:
from pyspark.sql.functions import count

df = df.groupBy("color").agg(count("*").alias("count"))

In [0]:
(df.writeStream.format("delta")
    .outputMode("Complete")
    .trigger(once=True)
    .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/checkpointLocation")
    .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/")
    .start() 
)

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/`

Phase 2 : Add more colours

In [0]:
%sql
insert into dbcertification.streaming.sourcetable values
('blue'),
('red'),
('green')
    

Run command again

In [0]:
(df.writeStream.format("delta")
    .outputMode("Complete")
    .trigger(once=True)
    .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/checkpointLocation")
    .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/")
    .start() 
)

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/`

Add one unique color now:

In [0]:
%sql
insert into dbcertification.streaming.sourcetable values
('purple')

In [0]:
(df.writeStream.format("delta")
    .outputMode("Complete")
    .trigger(once=True)
    .option("checkpointLocation","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/checkpointLocation")
    .option("path","abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/")
    .start() 
)

In [0]:
%sql
select * from delta.`abfss://streamingcontainer@dbcertificationstg.dfs.core.windows.net/deltaTable/sinktable/`

Update Mode is not supported in Delta Table

In [0]:
%skip
df.writeStream.format("delta")\
            .outputMode('update')\
            .trigger(once=True)\
            .option("checkpointLocation","/Volumes/workspace/stream/streaming/output/check")\
            .option("path","/Volumes/workspace/stream/streaming/output/Data")\
            .start()